# 5. Using Clustering Results for Analysis

This notebook demonstrates how to use the clustering results for practical applications.

## Use Cases
1. Query similar logs using FAISS
2. Detect anomalies based on cluster membership
3. Analyze temporal patterns within clusters
4. Generate alerts for unusual cluster behavior
5. Export cluster-based insights

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import faiss
from datetime import datetime, timedelta
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully")

## 1. Load Clustering Results

In [ ]:
# Load clustered data
df = pd.read_parquet('../output/grafana/clustered_logs.parquet')
embeddings = np.load('../output/grafana/embeddings.npy')
index = faiss.read_index('../output/grafana/faiss_index.bin')

with open('../output/grafana/cluster_summary.json', 'r') as f:
    cluster_summary = json.load(f)

print(f"✅ Loaded clustering results")
print(f"Dataset: {df.shape}")
print(f"Embeddings: {embeddings.shape}")
print(f"FAISS index: {index.ntotal:,} vectors")
print(f"Number of clusters: {cluster_summary.get('n_clusters', 'N/A')}")

## 2. Similarity Search with FAISS

In [ ]:
def find_similar_logs(query_text, top_k=10, use_embedding=False, query_idx=None):
    """
    Find similar logs using FAISS index.
    
    Args:
        query_text: Text to search for (if use_embedding=False)
        top_k: Number of results to return
        use_embedding: If True, use query_idx to get embedding
        query_idx: Index of log to use as query
    
    Returns:
        DataFrame with similar logs
    """
    if use_embedding and query_idx is not None:
        # Use existing embedding
        query_embedding = embeddings[query_idx:query_idx+1]
    else:
        # Generate new embedding (requires model)
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
        query_embedding = model.encode([query_text])
    
    # Normalize for cosine similarity
    query_embedding = query_embedding / np.linalg.norm(query_embedding)
    
    # Search
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    
    # Create results dataframe
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            'similarity': float(dist),
            'log_id': int(idx),
            'cluster': df.iloc[idx]['cluster'],
            'cluster_label': df.iloc[idx].get('cluster_label', 'unknown'),
            'service': df.iloc[idx]['service'],
            'dashboard': df.iloc[idx]['dashboard'],
            'panel_title': df.iloc[idx]['panel_title'],
            'text': df.iloc[idx]['normalized_text'][:150] + '...'
        })
    
    return pd.DataFrame(results)

# Example: Find logs similar to a specific log
query_idx = 1000
print(f"\n🔍 Query Log #{query_idx}:")
print(f"Text: {df.iloc[query_idx]['normalized_text'][:150]}...")
print(f"Service: {df.iloc[query_idx]['service']}")
print(f"Cluster: {df.iloc[query_idx]['cluster']}")

similar_logs = find_similar_logs(None, top_k=10, use_embedding=True, query_idx=query_idx)

print(f"\n📊 Top 10 Similar Logs:")
print(similar_logs[['log_id', 'similarity', 'cluster_label', 'service', 'panel_title']].to_string(index=False))

## 3. Anomaly Detection Based on Cluster Membership

In [ ]:
def detect_anomalies_by_cluster(df, embeddings, threshold=0.3):
    """
    Detect anomalous logs based on distance to cluster centroid.
    
    Args:
        df: DataFrame with cluster assignments
        embeddings: Log embeddings
        threshold: Distance threshold for anomaly detection
    
    Returns:
        DataFrame with anomaly scores
    """
    anomaly_scores = []
    
    for cluster_id in df[df['cluster'] != -1]['cluster'].unique():
        # Get cluster members
        cluster_mask = df['cluster'] == cluster_id
        cluster_embeddings = embeddings[cluster_mask]
        
        # Calculate centroid
        centroid = cluster_embeddings.mean(axis=0, keepdims=True)
        
        # Calculate distances to centroid
        distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
        
        # Normalize distances
        mean_dist = distances.mean()
        std_dist = distances.std()
        z_scores = (distances - mean_dist) / (std_dist + 1e-8)
        
        # Store scores
        cluster_indices = df[cluster_mask].index
        for idx, z_score in zip(cluster_indices, z_scores):
            anomaly_scores.append({
                'log_id': idx,
                'cluster': cluster_id,
                'anomaly_score': float(z_score),
                'is_anomaly': z_score > threshold
            })
    
    return pd.DataFrame(anomaly_scores)

# Detect anomalies
print("Detecting anomalies...")
anomalies_df = detect_anomalies_by_cluster(df, embeddings, threshold=2.5)

n_anomalies = anomalies_df['is_anomaly'].sum()
print(f"\n✅ Detected {n_anomalies:,} anomalous logs ({n_anomalies/len(anomalies_df)*100:.2f}%)")

# Show top anomalies
top_anomalies = anomalies_df.nlargest(10, 'anomaly_score')
print(f"\n🚨 Top 10 Anomalies:")
for _, row in top_anomalies.iterrows():
    log_id = int(row['log_id'])
    score = row['anomaly_score']
    cluster = row['cluster']
    print(f"\nLog #{log_id} (Cluster {cluster}, Score: {score:.2f})")
    print(f"  Service: {df.iloc[log_id]['service']}")
    print(f"  Panel: {df.iloc[log_id]['panel_title']}")
    print(f"  Text: {df.iloc[log_id]['normalized_text'][:100]}...")

## 4. Temporal Analysis by Cluster

In [ ]:
# Analyze temporal patterns for each cluster
if 'timestamp' in df.columns:
    # Convert timestamp if needed
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Group by cluster and hour
    df['hour'] = df['timestamp'].dt.hour
    
    # Select top 5 clusters by size
    top_clusters = df[df['cluster'] != -1]['cluster'].value_counts().head(5).index
    
    # Plot temporal patterns
    fig, axes = plt.subplots(len(top_clusters), 1, figsize=(14, 3*len(top_clusters)))
    if len(top_clusters) == 1:
        axes = [axes]
    
    for ax, cluster_id in zip(axes, top_clusters):
        cluster_df = df[df['cluster'] == cluster_id]
        hourly_counts = cluster_df.groupby('hour').size()
        
        cluster_label = cluster_df['cluster_label'].iloc[0] if 'cluster_label' in cluster_df else f'Cluster {cluster_id}'
        
        ax.plot(hourly_counts.index, hourly_counts.values, marker='o', linewidth=2)
        ax.fill_between(hourly_counts.index, hourly_counts.values, alpha=0.3)
        ax.set_title(f'Temporal Pattern: {cluster_label} (n={len(cluster_df):,})', fontweight='bold')
        ax.set_xlabel('Hour of Day')
        ax.set_ylabel('Log Count')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 23)
    
    plt.tight_layout()
    plt.savefig('../output/grafana/temporal_patterns_by_cluster.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Saved temporal patterns to: output/grafana/temporal_patterns_by_cluster.png")
else:
    print("⚠️  Timestamp information not available in dataset")

## 5. Cluster-Based Alert Generation

In [ ]:
def generate_alerts(df, window_minutes=10, spike_threshold=3.0):
    """
    Generate alerts based on cluster activity spikes.
    
    Args:
        df: DataFrame with cluster assignments and timestamps
        window_minutes: Time window for analysis
        spike_threshold: Standard deviations above mean for alert
    
    Returns:
        List of alerts
    """
    if 'timestamp' not in df.columns:
        print("⚠️  Timestamp required for alert generation")
        return []
    
    alerts = []
    
    for cluster_id in df[df['cluster'] != -1]['cluster'].unique():
        cluster_df = df[df['cluster'] == cluster_id]
        
        # Calculate activity over time windows
        cluster_df = cluster_df.set_index('timestamp')
        window_counts = cluster_df.resample(f'{window_minutes}T').size()
        
        # Calculate statistics
        mean_count = window_counts.mean()
        std_count = window_counts.std()
        
        # Detect spikes
        for timestamp, count in window_counts.items():
            z_score = (count - mean_count) / (std_count + 1e-8)
            
            if z_score > spike_threshold:
                cluster_label = cluster_summary['cluster_labels'].get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
                alerts.append({
                    'timestamp': timestamp,
                    'cluster_id': cluster_id,
                    'cluster_label': cluster_label,
                    'log_count': int(count),
                    'mean_count': float(mean_count),
                    'z_score': float(z_score),
                    'severity': 'high' if z_score > 5 else 'medium',
                    'message': f'Spike detected in {cluster_label}: {count} logs ({z_score:.1f}σ above normal)'
                })
    
    return sorted(alerts, key=lambda x: x['z_score'], reverse=True)

# Generate alerts
alerts = generate_alerts(df.reset_index(drop=True), window_minutes=30, spike_threshold=2.5)

print(f"\n🚨 Generated {len(alerts)} alerts")

if alerts:
    print(f"\nTop 10 Alerts:")
    for i, alert in enumerate(alerts[:10], 1):
        severity_emoji = '🔴' if alert['severity'] == 'high' else '🟡'
        print(f"\n{i}. {severity_emoji} {alert['timestamp'].strftime('%Y-%m-%d %H:%M')}")
        print(f"   {alert['message']}")
        print(f"   Expected: {alert['mean_count']:.0f}, Observed: {alert['log_count']}")

## 6. Export Cluster-Based Insights

In [ ]:
def export_cluster_profiles(df, output_file):
    """
    Export comprehensive cluster profiles for each cluster.
    """
    profiles = []
    
    for cluster_id in sorted(df[df['cluster'] != -1]['cluster'].unique()):
        cluster_df = df[df['cluster'] == cluster_id]
        
        profile = {
            'cluster_id': int(cluster_id),
            'cluster_label': cluster_df['cluster_label'].iloc[0] if 'cluster_label' in cluster_df else f'Cluster {cluster_id}',
            'size': len(cluster_df),
            'percentage': float(len(cluster_df) / len(df) * 100),
            'top_services': cluster_df['service'].value_counts().head(5).to_dict(),
            'top_dashboards': cluster_df['dashboard'].value_counts().head(5).to_dict(),
            'top_panels': cluster_df['panel_title'].value_counts().head(5).to_dict(),
            'panel_types': cluster_df['panel_type'].value_counts().to_dict(),
            'sample_logs': cluster_df['normalized_text'].head(3).tolist()
        }
        
        # Add value statistics if available
        if 'value' in cluster_df.columns:
            values = cluster_df['value'].dropna()
            if len(values) > 0:
                profile['value_stats'] = {
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'min': float(values.min()),
                    'max': float(values.max())
                }
        
        profiles.append(profile)
    
    # Save to file
    with open(output_file, 'w') as f:
        json.dump({
            'generation_time': datetime.now().isoformat(),
            'total_clusters': len(profiles),
            'total_logs': len(df),
            'profiles': profiles
        }, f, indent=2)
    
    return profiles

# Export cluster profiles
profiles_file = '../output/grafana/cluster_profiles.json'
profiles = export_cluster_profiles(df, profiles_file)

print(f"\n✅ Exported {len(profiles)} cluster profiles to: {profiles_file}")

# Display summary
print(f"\n📊 Cluster Profiles Summary:")
for profile in profiles[:5]:  # Show first 5
    print(f"\n{profile['cluster_label']} (n={profile['size']:,}, {profile['percentage']:.1f}%)")
    print(f"  Top Service: {list(profile['top_services'].keys())[0]}")
    print(f"  Top Dashboard: {list(profile['top_dashboards'].keys())[0]}")

## 7. Query Interface for Cluster Exploration

In [ ]:
def explore_cluster(cluster_id, df, max_samples=10):
    """
    Interactive cluster exploration.
    """
    cluster_df = df[df['cluster'] == cluster_id]
    
    if len(cluster_df) == 0:
        print(f"⚠️  No logs found in cluster {cluster_id}")
        return
    
    print("\n" + "="*80)
    print(f"CLUSTER {cluster_id} EXPLORATION")
    print("="*80)
    
    cluster_label = cluster_df['cluster_label'].iloc[0] if 'cluster_label' in cluster_df else f'Cluster {cluster_id}'
    print(f"\n🏷️  Label: {cluster_label}")
    print(f"📊 Size: {len(cluster_df):,} logs ({len(cluster_df)/len(df)*100:.2f}% of total)")
    
    print(f"\n📋 Top 5 Services:")
    for service, count in cluster_df['service'].value_counts().head(5).items():
        print(f"  {service}: {count:,} ({count/len(cluster_df)*100:.1f}%)")
    
    print(f"\n📊 Top 5 Dashboards:")
    for dashboard, count in cluster_df['dashboard'].value_counts().head(5).items():
        print(f"  {dashboard}: {count:,} ({count/len(cluster_df)*100:.1f}%)")
    
    print(f"\n📐 Panel Types:")
    for ptype, count in cluster_df['panel_type'].value_counts().items():
        print(f"  {ptype}: {count:,} ({count/len(cluster_df)*100:.1f}%)")
    
    if 'value' in cluster_df.columns:
        values = cluster_df['value'].dropna()
        if len(values) > 0:
            print(f"\n📈 Metric Values:")
            print(f"  Mean: {values.mean():.2f}")
            print(f"  Median: {values.median():.2f}")
            print(f"  Range: [{values.min():.2f}, {values.max():.2f}]")
    
    print(f"\n📝 Sample Logs (showing {min(max_samples, len(cluster_df))}):")
    for i, (_, row) in enumerate(cluster_df.head(max_samples).iterrows(), 1):
        print(f"\n{i}. {row['service']} / {row['panel_title']}")
        print(f"   {row['normalized_text'][:120]}...")
    
    print("\n" + "="*80)

# Example: Explore a specific cluster
example_cluster = df[df['cluster'] != -1]['cluster'].value_counts().index[0]
explore_cluster(example_cluster, df)

## 8. Summary and Export

In [ ]:
# Create comprehensive usage report
usage_report = {
    'timestamp': datetime.now().isoformat(),
    'dataset_stats': {
        'total_logs': len(df),
        'total_clusters': int(df[df['cluster'] != -1]['cluster'].nunique()),
        'noise_logs': int((df['cluster'] == -1).sum())
    },
    'anomaly_detection': {
        'anomalies_detected': int(n_anomalies) if 'n_anomalies' in locals() else 0,
        'anomaly_rate': float(n_anomalies / len(df) * 100) if 'n_anomalies' in locals() else 0
    },
    'alerts_generated': {
        'total_alerts': len(alerts) if 'alerts' in locals() else 0,
        'high_severity': sum(1 for a in alerts if a['severity'] == 'high') if 'alerts' in locals() else 0
    },
    'exports': {
        'cluster_profiles': profiles_file,
        'visualizations': [
            '../output/grafana/temporal_patterns_by_cluster.png'
        ]
    }
}

usage_report_file = '../output/grafana/usage_report.json'
with open(usage_report_file, 'w') as f:
    json.dump(usage_report, f, indent=2)

print("\n" + "="*80)
print("USAGE SUMMARY")
print("="*80)
print(json.dumps(usage_report, indent=2))
print("\n" + "="*80)
print(f"\n✅ Usage report saved to: {usage_report_file}")

print("\n🎉 Successfully demonstrated clustering results usage!")
print("\nKey Capabilities:")
print("  ✅ Similarity search with FAISS")
print("  ✅ Anomaly detection based on cluster membership")
print("  ✅ Temporal pattern analysis")
print("  ✅ Alert generation for cluster spikes")
print("  ✅ Comprehensive cluster profiling")
print("  ✅ Interactive cluster exploration")